[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/18_embedding.ipynb)

# 🟢 Easy: Embedding Layer

Implement an **embedding lookup table** from scratch.

### Signature
```python
class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int): ...
    def forward(self, indices: Tensor) -> Tensor: ...
```

### Rules
- `self.weight`: `nn.Parameter` of shape `(num_embeddings, embedding_dim)`
- Forward: index into weight matrix — `weight[indices]`
- Do NOT use `nn.Embedding`

In [8]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [9]:
import torch
import torch.nn as nn

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

    def forward(self, indices):
        return self.weight[indices]

In [11]:
# 🧪 Debug
emb = MyEmbedding(10, 4)
idx = torch.tensor([0, 3, 7])
print('Output shape:', emb(idx).shape)
print('Matches manual:', torch.equal(emb(idx)[0], emb.weight[0]))

Output shape: torch.Size([3, 4])
Matches manual: True


In [12]:
# ✅ SUBMIT
from torch_judge import check
check('embedding')


🧪 Testing: Embedding Layer (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Weight shape (0.5ms)
  ✅ [2/4] Lookup correctness (0.6ms)
  ✅ [3/4] Batch of indices (0.3ms)
  ✅ [4/4] Gradient flow (2.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (3.4ms total)
  Progress saved. Run status() to see your dashboard.



In [16]:
import torch
import torch.nn as nn

class EmbeddingFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, indices):
        # weight: (num_embeddings, embedding_dim)
        # indices: 任意整数张量,比如 (B, L)
        ctx.save_for_backward(indices)
        ctx.num_embeddings = weight.shape[0]
        return weight[indices]   # 输出: (B, L, embedding_dim)

    @staticmethod
    def backward(ctx, grad_output):
        # grad_output 形状和 forward 输出一致: (B, L, D)
        indices, = ctx.saved_tensors
        N = ctx.num_embeddings
        D = grad_output.shape[-1]

        # 拍平: 任意形状的 indices -> 一维; grad -> 二维
        flat_idx  = indices.reshape(-1)          # (B*L,)
        flat_grad = grad_output.reshape(-1, D)   # (B*L, D)

        # 关键: index_add_ 把每行 grad 累加到对应的 W[i]
        grad_weight = torch.zeros(N, D,
                                  dtype=grad_output.dtype,
                                  device=grad_output.device)
        grad_weight.index_add_(0, flat_idx, flat_grad)

        # indices 是整数不需要梯度; weight 需要
        return grad_weight, None


class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_embeddings, embedding_dim))

    def forward(self, indices):
        return EmbeddingFunction.apply(self.weight, indices)

torch.manual_seed(0)
N, D = 10, 4

my  = MyEmbedding(N, D)
ref = nn.Embedding(N, D)
ref.weight.data.copy_(my.weight.data)

# 故意让索引有重复,测试梯度累加
idx = torch.tensor([[2, 5, 2],
                    [7, 2, 5]])

out_my, out_ref = my(idx), ref(idx)
print("forward 一致:", torch.allclose(out_my, out_ref))

g = torch.randn_like(out_my)
out_my.backward(g)
out_ref.backward(g)
print("backward 一致:", torch.allclose(my.weight.grad, ref.weight.grad))

# W[2] 出现了 3 次: (0,0), (0,2), (1,1)
print("W[2] grad =", my.weight.grad[2])
print("手算累加  =", g[0,0] + g[0,2] + g[1,1])   # 应该相等

forward 一致: True
backward 一致: True
W[2] grad = tensor([ 1.2987, -1.4237, -0.4859, -1.8417])
手算累加  = tensor([ 1.2987, -1.4237, -0.4859, -1.8417])
